# Max-Cut on a large graph with the 4colors Optimizer for QUBO

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/4colors-research/4colors-examples/blob/main/notebooks/qubo-maxcut.ipynb)

Write Max-Cut as a QUBO, watch better cuts arrive, and check the answer yourself.
Runs with an API key, or offline (no key) with a small local stand-in whose answers are **not** 4colors results.

In [ ]:
%pip install -q "fourcolors[optimizer]"

## Key or offline

Set `OFFLINE = True` to run without a key. With a key, store it as a Colab secret named `FOURCOLORS_API_KEY`
(the key icon on the left) or set the environment variable before starting Jupyter. Never paste a key into a cell.

In [ ]:
import os

OFFLINE = not os.environ.get("FOURCOLORS_API_KEY")
try:                                    # Colab secrets, when running on Colab
    from google.colab import userdata
    os.environ["FOURCOLORS_API_KEY"] = userdata.get("FOURCOLORS_API_KEY")
    OFFLINE = False
except Exception:
    pass

import fourcolors
import fourcolors.qubo

if OFFLINE:
    # A tiny local stand-in (the same as shared/examplekit.py's OfflineQubo, enumeration only), for trying the code.
    import itertools
    from types import SimpleNamespace

    def _offline_solve(Q, time_limit_s=1, linear=None, **kw):
        n = len(linear)
        best = min(itertools.product((0, 1), repeat=n),
                   key=lambda x: sum(v * x[i] * x[j] for (i, j), v in Q.items()) + sum(l * a for l, a in zip(linear, x)))
        obj = sum(v * best[i] * best[j] for (i, j), v in Q.items()) + sum(l * a for l, a in zip(linear, best))
        return SimpleNamespace(status="offline", objective=obj, solution=list(best),
                               trajectory=[{"t_search_s": 0.0, "objective": obj}])
    fourcolors.qubo.solve = _offline_solve
print("offline" if OFFLINE else "using the service")

## Write the problem as a QUBO

Edge (i, j) is cut when x<sub>i</sub> + x<sub>j</sub> − 2 x<sub>i</sub> x<sub>j</sub> = 1, so minimizing
E(x) = Σ<sub>edges</sub> (2 x<sub>i</sub> x<sub>j</sub> − x<sub>i</sub> − x<sub>j</sub>) maximizes the cut, and the cut size is −E.

In [ ]:
import random

n, m = (14, 30) if OFFLINE else (2000, 6000)
rnd = random.Random(7)
edges = set()
while len(edges) < m:
    i, j = rnd.sample(range(n), 2)
    edges.add((min(i, j), max(i, j)))

Q = {e: 2 for e in edges}
linear = [0] * n
for i, j in edges:
    linear[i] -= 1
    linear[j] -= 1
print(n, "nodes,", len(edges), "edges")

## Solve it

`time_limit_s` is the search time you pay for. With the service, `fourcolors.qubo.submit` and
`job.incumbents()` stream each better cut as it is found (see `tutorials/qubo/maxcut/maxcut.py`).

In [ ]:
result = fourcolors.qubo.solve(Q, time_limit_s=1 if OFFLINE else 30, linear=linear)
print(result.status, "cut", -result.objective)

## Check the answer yourself

In [ ]:
x = result.solution
cut = sum(1 for i, j in edges if x[i] != x[j])
print(f"{cut} edges cross between groups of {x.count(0)} and {x.count(1)} nodes")
assert cut == -result.objective
for step in result.trajectory[-5:]:
    print(f"t={step['t_search_s']:.2f} s  objective {step['objective']}")

## Next

- Large models: upload once, reuse: [docs](https://docs.4colors-research.com/start/tutorials/qubo-large-models/?utm_source=github&utm_medium=examples&utm_campaign=qubo-maxcut&utm_content=notebook).
- Get an API key: [account.4colors-research.com](https://account.4colors-research.com/?utm_source=github&utm_medium=examples&utm_campaign=qubo-maxcut&utm_content=notebook).